# Figure 4: Negative slope recovery

This notebook summarizes slope-sign transitions between RiverSP and filtered PIXC at the matched reach-overpass level. Positive recovery is interpreted as improved physical plausibility and potential usability in discharge workflows, not as independent validation of absolute slope accuracy.

In [ ]:
2+2

In [ ]:
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap

INK = "#17222B"
MUTED = "#647078"
GRID = "#DCE2E5"
BLUE = "#1976A8"
ORANGE = "#C17C32"
IMPROVE = "#278F86"
NEGATIVE = "#C65A64"
NEUTRAL = "#EEF2F3"

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans"],
    "font.size": 9,
    "axes.labelsize": 9,
    "axes.labelcolor": INK,
    "axes.edgecolor": INK,
    "axes.linewidth": 0.8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "figure.dpi": 160,
    "savefig.dpi": 600,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

In [ ]:
def find_project_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "data").is_dir() and (candidate / "raqw").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the project root.")


PROJECT_ROOT = find_project_root()
MATCHED_CSV = (
    PROJECT_ROOT / "data" / "riversp_comparison_updated_filter"
    / "matched_pixc_riversp_slopes.csv"
)
FIGURE_DIR = PROJECT_ROOT / "publication_figs" / "outputs"
OUTPUT_STEM = "figure_04_negative_slope_recovery"
EXPORT_FILES = False
if EXPORT_FILES:
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)

assert MATCHED_CSV.exists(), MATCHED_CSV

In [ ]:
observations = pd.read_csv(MATCHED_CSV, dtype={"reach_id": str})
filtered_col = "updated_postfilter_slope_m_per_km"
product_columns = {
    "RiverSP slope": "riversp_slope_m_per_km",
    "RiverSP slope2": "riversp_slope2_m_per_km",
}

# Use one common complete-case table so slope and slope2 have the same denominator.
paired = observations.dropna(subset=[filtered_col, *product_columns.values()]).copy()
filtered = paired[filtered_col]


def transition_table(riversp_column):
    riversp = paired[riversp_column]
    counts = np.array([
        [((riversp < 0) & (filtered < 0)).sum(), ((riversp < 0) & (filtered > 0)).sum()],
        [((riversp > 0) & (filtered < 0)).sum(), ((riversp > 0) & (filtered > 0)).sum()],
    ], dtype=int)
    percentages = counts / counts.sum(axis=1, keepdims=True) * 100
    return counts, percentages


transition_results = {
    label: transition_table(column) for label, column in product_columns.items()
}

for label, column in product_columns.items():
    counts, percentages = transition_results[label]
    negative_mask = paired[column] < 0
    recovered = filtered[negative_mask]
    print(f"{label}:")
    print(f"  Negative observations: {negative_mask.sum():,}")
    print(f"  Negative to positive: {counts[0, 1]:,} ({percentages[0, 1]:.1f}%)")
    print(f"  Negative to negative: {counts[0, 0]:,} ({percentages[0, 0]:.1f}%)")
    print(f"  Reaches with a negative observation: {paired.loc[negative_mask, 'reach_id'].nunique():,}")
    print(f"  Median filtered slope for negative cases: {recovered.median():.3f} m km^-1")

print(f"Common complete reach-overpasses: {len(paired):,}")
print(f"Unique reaches: {paired['reach_id'].nunique():,}")

In [ ]:
def panel_heading(ax, letter, title):
    ax.text(
        0, 1.035, f"{letter}   {title}", transform=ax.transAxes,
        ha="left", va="bottom", fontsize=10.5, fontweight="bold", color=INK,
    )


def style_axis(ax):
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_axisbelow(True)


fig = plt.figure(figsize=(10.6, 6.4), facecolor="white")
outer = fig.add_gridspec(
    2, 2, height_ratios=[0.30, 1.0], width_ratios=[1.12, 0.88],
    left=0.075, right=0.985, bottom=0.11, top=0.94,
    wspace=0.20, hspace=0.30,
)

# Panel A: compact recovery-rate summary for both RiverSP slope products.
ax_recovery = fig.add_subplot(outer[0, :])
recovery_labels = ["RiverSP $slope$", "RiverSP $slope2$"]
recovery_counts = [transition_results["RiverSP slope"][0][0, 1], transition_results["RiverSP slope2"][0][0, 1]]
negative_counts = [transition_results["RiverSP slope"][0][0].sum(), transition_results["RiverSP slope2"][0][0].sum()]
recovery_rates = np.array(recovery_counts) / np.array(negative_counts) * 100
bar_y = np.array([1, 0])
ax_recovery.barh(bar_y, [100, 100], color="#EDF1F2", height=0.42, edgecolor="none")
ax_recovery.barh(bar_y, recovery_rates, color=IMPROVE, height=0.42, edgecolor="none")
for y_position, recovered, total, rate in zip(bar_y, recovery_counts, negative_counts, recovery_rates):
    ax_recovery.text(
        min(rate - 1.2, 96), y_position, f"{recovered:,} / {total:,}   ({rate:.1f}%)",
        ha="right", va="center", fontsize=8.5, fontweight="bold", color="white",
    )
ax_recovery.set_xlim(0, 100)
ax_recovery.set_yticks(bar_y, recovery_labels)
ax_recovery.set_xticks([0, 50, 100])
ax_recovery.set_xlabel("Negative RiverSP cases recovered as positive (%)")
ax_recovery.grid(axis="x", color=GRID, linewidth=0.5)
ax_recovery.spines[["top", "right", "left"]].set_visible(False)
ax_recovery.tick_params(axis="y", length=0, pad=8)
panel_heading(ax_recovery, "a", "Negative RiverSP slopes recovered as positive")

# Panel B: primary scatterplot for observations with negative RiverSP slope.
ax_scatter = fig.add_subplot(outer[1, 0])
slope_col = product_columns["RiverSP slope"]
negative_slope = paired[slope_col] < 0
x = paired.loc[negative_slope, slope_col].to_numpy(float)
y = paired.loc[negative_slope, filtered_col].to_numpy(float)
recovered_mask = y > 0
ax_scatter.axhspan(0, 32, color=IMPROVE, alpha=0.07, linewidth=0, zorder=0)
ax_scatter.scatter(
    x[~recovered_mask], y[~recovered_mask], s=13, color=NEGATIVE,
    alpha=0.55, edgecolor="white", linewidth=0.25, rasterized=True,
    label="Remains negative", zorder=2,
)
ax_scatter.scatter(
    x[recovered_mask], y[recovered_mask], s=12, color=IMPROVE,
    alpha=0.38, edgecolor="none", rasterized=True,
    label="Recovered positive", zorder=2,
)
upper_left_mask = (x < 0) & (y > 0)
line_x = np.array([-31, 0])
ax_scatter.plot(
    line_x, -line_x, color=INK, linewidth=1.1, linestyle=(0, (4, 2)),
    label="$y=-x$", zorder=3,
)
if upper_left_mask.sum() > 1:
    observed = y[upper_left_mask]
    predicted = -x[upper_left_mask]
    ss_res = np.sum((observed - predicted) ** 2)
    ss_tot = np.sum((observed - observed.mean()) ** 2)
    line_r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
    ax_scatter.text(
        -26.2, 27.8, f"$R^2$ = {line_r2:.2f}",
        ha="left", va="center", fontsize=8.2, color=INK,
        bbox={"boxstyle": "round,pad=0.18", "facecolor": "white", "edgecolor": "none", "alpha": 0.78},
    )
ax_scatter.axhline(0, color=INK, linewidth=0.9)
ax_scatter.axvline(0, color=INK, linewidth=0.9)
ax_scatter.set_xlim(-33, 0.8)
ax_scatter.set_ylim(-36, 31)
ax_scatter.set_xlabel("RiverSP $slope$ (m km$^{-1}$)")
ax_scatter.set_ylabel("Filtered PIXC slope (m km$^{-1}$)")
ax_scatter.grid(color=GRID, linewidth=0.5)
ax_scatter.legend(loc="lower left", frameon=False, fontsize=7.5)
panel_heading(ax_scatter, "b", "Filtered slope when RiverSP slope is negative")
style_axis(ax_scatter)

# Panel C: filtered-slope magnitudes for the primary RiverSP slope subset only.
ax_hist = fig.add_subplot(outer[1, 1])
negative_filtered = filtered.loc[negative_slope].to_numpy(float)
display_min, display_max = -5, 20
display_values = negative_filtered[(negative_filtered >= display_min) & (negative_filtered <= display_max)]
# Anchor the bin edges at zero so the first nonnegative bar begins at x = 0.
bin_width = 0.75
bins = np.concatenate([
    np.arange(-5.25, 0, bin_width),
    np.arange(0, 20.25 + bin_width, bin_width),
])
ax_hist.hist(
    display_values, bins=bins, color=BLUE, alpha=0.72,
    edgecolor="white", linewidth=0.6,
)
ax_hist.axvspan(0, display_max, color=IMPROVE, alpha=0.025, linewidth=0, zorder=0)
ax_hist.axvline(0, color=INK, linewidth=0.9)
median_filtered = np.median(negative_filtered)
ax_hist.axvline(median_filtered, color=DARK_BLUE if "DARK_BLUE" in globals() else BLUE, linewidth=1.3, linestyle=(0, (4, 2)))
ax_hist.text(
    0.97, 0.94,
    f"Median = {median_filtered:.2f} m km$^{{-1}}$",
    transform=ax_hist.transAxes, color=BLUE,
    ha="right", va="top", fontsize=8, fontweight="bold",
)
ax_hist.text(
    0.97, 0.77,
    f"{len(negative_filtered) - len(display_values)} values outside plotted range",
    transform=ax_hist.transAxes, ha="right", va="top", fontsize=7.2, color=MUTED,
)
ax_hist.set_xlim(display_min, display_max)
ax_hist.set_xlabel("Filtered PIXC slope (m km$^{-1}$)")
ax_hist.set_ylabel("Number of reach-overpasses")
ax_hist.grid(axis="y", color=GRID, linewidth=0.5)
panel_heading(ax_hist, "c", "Filtered slope magnitudes when RiverSP slope is negative")
style_axis(ax_hist)

if EXPORT_FILES:
    png_path = FIGURE_DIR / f"{OUTPUT_STEM}.png"
    pdf_path = FIGURE_DIR / f"{OUTPUT_STEM}.pdf"
    fig.savefig(png_path, bbox_inches="tight", facecolor="white")
    fig.savefig(pdf_path, bbox_inches="tight", facecolor="white")
    print(f"Saved {png_path}")
    print(f"Saved {pdf_path}")

plt.show()

In [ ]:
# Exploratory diagnostic: profile gallery for near-opposite recovered slopes.
# This cell is not part of the official Figure 4.
from io import StringIO
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import json
import urllib.error
import urllib.parse
import urllib.request

RIVERSP_CACHE_CSV = PROJECT_ROOT / "data" / "riversp_comparison_updated_filter" / "riversp_daily_cache.csv"
SWORD_NODES_SHP = Path(r"C:\UNESCO\Code\data\shp\SA\sa_sword_nodes_hb66_v17b.shp")
HYDROCRON_URL = "https://soto.podaac.earthdatacloud.nasa.gov/hydrocron/v1/timeseries"
NODE_COLLECTION = "SWOT_L2_HR_RiverSP_node_D"
NODE_FIELDS = "reach_id,node_id,time_str,wse,node_q,p_dist_out,geoid_hght"
NODE_CACHE_DIR = PROJECT_ROOT / "publication_figs" / "outputs" / "hydrocron_node_cache"
NODE_FETCH_WORKERS = 8

riversp_cache = pd.read_csv(RIVERSP_CACHE_CSV, dtype={"reach_id": str})
_node_profile_cache = {}
_sword_nodes_cache = {}
_node_warning_messages = set()


def warn_once(message):
    if message not in _node_warning_messages:
        print(message)
        _node_warning_messages.add(message)


def hydrocron_response_to_df(text):
    payload = json.loads(text)
    csv_text = payload.get("results", {}).get("csv", "")
    if not csv_text.strip():
        return pd.DataFrame()
    return pd.read_csv(StringIO(csv_text))


def load_sword_nodes_for_reach(reach_id):
    reach_key = str(reach_id)
    if reach_key in _sword_nodes_cache:
        return _sword_nodes_cache[reach_key].copy()

    try:
        import geopandas as gpd
    except ImportError as exc:
        raise ImportError("geopandas is required to read SWORD node IDs for Hydrocron node retrieval") from exc

    nodes = gpd.read_file(SWORD_NODES_SHP, where=f"reach_id = {int(reach_id)}")
    if nodes.empty:
        return pd.DataFrame()

    out = pd.DataFrame(nodes.drop(columns="geometry"))
    out["node_id"] = out["node_id"].astype("int64").astype(str)
    out["reach_id"] = out["reach_id"].astype("int64").astype(str)
    out["dist_out"] = pd.to_numeric(out["dist_out"], errors="coerce")
    out["node_s_m"] = out["dist_out"] - out["dist_out"].min()
    out = out.sort_values("node_s_m").reset_index(drop=True)
    _sword_nodes_cache[reach_key] = out
    return out.copy()


def fetch_hydrocron_node(node_id, start_time, end_time):
    params = {
        "feature": "Node",
        "feature_id": str(node_id),
        "start_time": start_time,
        "end_time": end_time,
        "output": "csv",
        "collection_name": NODE_COLLECTION,
        "fields": NODE_FIELDS,
    }
    url = HYDROCRON_URL + "?" + urllib.parse.urlencode(params)
    with urllib.request.urlopen(url, timeout=20) as response:
        return hydrocron_response_to_df(response.read().decode("utf-8"))


def load_hydrocron_nodes_for_example(row):
    cache_key = (str(row["reach_id"]), str(row["riversp_time_utc"]))
    if cache_key in _node_profile_cache:
        return _node_profile_cache[cache_key].copy()
    safe_time = pd.to_datetime(row["riversp_time_utc"], utc=True).strftime("%Y%m%dT%H%M%SZ")
    cache_path = NODE_CACHE_DIR / f"reach_{cache_key[0]}_{safe_time}_hydrocron_nodes_ellipsoid.csv"
    if cache_path.exists():
        cached = pd.read_csv(cache_path, dtype={"reach_id": str, "node_id": str})
        _node_profile_cache[cache_key] = cached
        return cached.copy()

    try:
        sword_nodes = load_sword_nodes_for_reach(row["reach_id"])
    except Exception as exc:
        warn_once(f"Skipping Hydrocron nodes: {exc}")
        _node_profile_cache[cache_key] = pd.DataFrame()
        return pd.DataFrame()

    if sword_nodes.empty:
        warn_once(f"No SWORD nodes found for reach {row['reach_id']}.")
        _node_profile_cache[cache_key] = pd.DataFrame()
        return pd.DataFrame()

    riversp_time = pd.to_datetime(row["riversp_time_utc"], utc=True)
    start_time = (riversp_time - pd.Timedelta(minutes=45)).strftime("%Y-%m-%dT%H:%M:%SZ")
    end_time = (riversp_time + pd.Timedelta(minutes=45)).strftime("%Y-%m-%dT%H:%M:%SZ")

    print(f"Fetching Hydrocron nodes for reach {row['reach_id']} ({len(sword_nodes)} nodes)...")
    records = []
    node_ids = sword_nodes["node_id"].tolist()
    with ThreadPoolExecutor(max_workers=NODE_FETCH_WORKERS) as pool:
        futures = {
            pool.submit(fetch_hydrocron_node, node_id, start_time, end_time): node_id
            for node_id in node_ids
        }
        for future in as_completed(futures):
            node_id = futures[future]
            try:
                node_df = future.result()
            except urllib.error.HTTPError:
                node_df = pd.DataFrame()
            except Exception as exc:
                warn_once(f"Hydrocron node retrieval failed for reach {row['reach_id']} node {node_id}: {exc}")
                node_df = pd.DataFrame()
            if not node_df.empty:
                records.append(node_df)

    if records:
        hydro_nodes = pd.concat(records, ignore_index=True)
        hydro_nodes["node_id"] = hydro_nodes["node_id"].astype("int64").astype(str)
        hydro_nodes["reach_id"] = hydro_nodes["reach_id"].astype("int64").astype(str)
        for column in ["wse", "node_q", "p_dist_out", "geoid_hght"]:
            hydro_nodes[column] = pd.to_numeric(hydro_nodes[column], errors="coerce")
        hydro_nodes.loc[hydro_nodes["wse"] < -1.0e10, "wse"] = np.nan

        # If Hydrocron returns multiple passes inside the window, keep the pass nearest to this RiverSP time.
        hydro_nodes["time_dt"] = pd.to_datetime(hydro_nodes["time_str"], utc=True, errors="coerce")
        if hydro_nodes["time_dt"].notna().any():
            nearest_time = hydro_nodes.groupby("time_str")["time_dt"].first().sub(riversp_time).abs().idxmin()
            hydro_nodes = hydro_nodes.loc[hydro_nodes["time_str"] == nearest_time].copy()
    else:
        hydro_nodes = pd.DataFrame(columns=NODE_FIELDS.split(","))

    merged = sword_nodes[["node_id", "reach_id", "dist_out", "node_s_m"]].merge(
        hydro_nodes,
        on=["node_id", "reach_id"],
        how="left",
    )
    merged["node_s_m"] = merged["p_dist_out"].fillna(merged["dist_out"]) - sword_nodes["dist_out"].min()
    merged["wse_ellipsoid"] = merged["wse"] + merged["geoid_hght"]
    merged = merged.sort_values("node_s_m").reset_index(drop=True)
    NODE_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    merged.to_csv(cache_path, index=False)
    _node_profile_cache[cache_key] = merged
    return merged.copy()


recovered = paired[(paired["riversp_slope_m_per_km"] < 0) & (paired[filtered_col] > 0)].copy()
recovered["opposite_abs_resid"] = (recovered["riversp_slope_m_per_km"] + recovered[filtered_col]).abs()
recovered["opposite_rel_resid"] = recovered["opposite_abs_resid"] / recovered[
    ["riversp_slope_m_per_km", filtered_col]
].abs().max(axis=1)
recovered["opposite_magnitude"] = recovered[
    ["riversp_slope_m_per_km", filtered_col]
].abs().min(axis=1)

# Prefer examples with visible slope magnitude and one case per reach so the gallery is not redundant.
examples = []
seen_reaches = set()
for _, row in recovered.sort_values(["opposite_abs_resid", "opposite_rel_resid"]).iterrows():
    reach_id = row["reach_id"]
    if reach_id in seen_reaches or row["opposite_magnitude"] < 0.5:
        continue
    year = int(row["year"])
    stem = Path(row["file"]).stem
    points_csv = (
        PROJECT_ROOT / "data" / "chile_reaches_with_valid_discharge_run" / "processed"
        / f"reach_{reach_id}" / str(year) / "pixc_points" / f"{stem}_pts.csv"
    )
    updated_csv = (
        PROJECT_ROOT / "data" / "tau_range_update_experiment"
        / f"reach_{reach_id}" / str(year) / "updated_tau_window_results.csv"
    )
    if points_csv.exists() and updated_csv.exists():
        examples.append((row, points_csv, updated_csv))
        seen_reaches.add(reach_id)
    if len(examples) == 6:
        break

if len(examples) < 1:
    raise RuntimeError("No profile examples with available point/filter files were found.")

fig_profiles, axes = plt.subplots(2, 3, figsize=(12.0, 7.0), sharey=False)
axes = axes.ravel()
fig_profiles.subplots_adjust(left=0.07, right=0.985, bottom=0.10, top=0.90, wspace=0.22, hspace=0.34)

for ax, (row, points_csv, updated_csv) in zip(axes, examples):
    pts = pd.read_csv(points_csv).dropna(subset=["s_m", "height"]).sort_values("s_m")
    updated_rows = pd.read_csv(updated_csv)
    updated_row = updated_rows.loc[updated_rows["file"] == row["file"]].iloc[0]
    nodes = load_hydrocron_nodes_for_example(row)
    valid_nodes = nodes.loc[nodes["wse_ellipsoid"].notna()].copy() if "wse_ellipsoid" in nodes else pd.DataFrame()
    if not valid_nodes.empty and "node_q" in valid_nodes:
        valid_nodes = valid_nodes.loc[pd.to_numeric(valid_nodes["node_q"], errors="coerce").fillna(9) <= 1].copy()

    x_m = pts["s_m"].to_numpy(float)
    x_km = x_m / 1000.0
    y_wse = pts["height"].to_numpy(float)

    tau_low = float(updated_row["updated_tau_low"])
    tau_high = float(updated_row["updated_tau_high"])
    detrend_slope = float(updated_row["updated_detrend_slope_m_per_km"]) / 1000.0
    detrended = y_wse - detrend_slope * x_m
    lower_height = np.quantile(detrended, tau_low)
    upper_height = np.quantile(detrended, tau_high)
    keep = (detrended >= lower_height) & (detrended <= upper_height)

    raw_slope, raw_intercept = np.polyfit(x_m, y_wse, 1)
    filtered_slope, filtered_intercept = np.polyfit(x_m[keep], y_wse[keep], 1)
    line_x_m = np.linspace(x_m.min(), x_m.max(), 250)
    line_x_km = line_x_m / 1000.0
    raw_line = raw_intercept + raw_slope * line_x_m
    filtered_line = filtered_intercept + filtered_slope * line_x_m

    cache_match = riversp_cache.loc[
        (riversp_cache["reach_id"] == row["reach_id"])
        & (riversp_cache["time_utc"] == row["riversp_time_utc"])
    ]
    if cache_match.empty:
        riversp_wse = np.nanmedian(y_wse)
    else:
        riversp_wse = float(cache_match.iloc[0]["wse"])
    if not valid_nodes.empty and valid_nodes["geoid_hght"].notna().any():
        # RiverSP reach WSE is geoid-referenced; add node geoid height to compare with ellipsoidal PIXC heights.
        riversp_wse = riversp_wse + float(valid_nodes["geoid_hght"].median())
    # The plotted distance axis follows SWORD dist_out / PIXC s_m and increases upstream.
    # Hydrocron reports signed RiverSP slopes in the downstream direction, so flip the sign for display on this axis.
    riversp_reported_slope_m_per_km = float(row["riversp_slope_m_per_km"])
    riversp_plot_slope_m_per_km = -riversp_reported_slope_m_per_km
    riversp_line = riversp_wse + (riversp_plot_slope_m_per_km / 1000.0) * (line_x_m - x_m.mean())
    riversp_reported_slope2_m_per_km = float(row["riversp_slope2_m_per_km"])
    riversp_plot_slope2_m_per_km = -riversp_reported_slope2_m_per_km
    riversp_slope2_line = riversp_wse + (riversp_plot_slope2_m_per_km / 1000.0) * (line_x_m - x_m.mean())

    ax.scatter(x_km, y_wse, s=7, color="#7D878C", alpha=0.18, linewidths=0, rasterized=True, label="Raw PIXC")
    ax.scatter(x_km[~keep], y_wse[~keep], s=8, marker="x", color="#9FA8AD", alpha=0.55, linewidths=0.45, rasterized=True, label="Rejected")
    ax.scatter(x_km[keep], y_wse[keep], s=8, color=BLUE, alpha=0.62, linewidths=0, rasterized=True, label="Retained")
    ax.plot(line_x_km, filtered_line, color=BLUE, linewidth=1.8, label="Filtered OLS")
    ax.plot(line_x_km, riversp_line, color=NEGATIVE, linewidth=1.35, linestyle=(0, (5, 3)), label="RiverSP slope, plotted")
    ax.plot(line_x_km, riversp_slope2_line, color=ORANGE, linewidth=1.35, linestyle=(0, (2, 2)), label="RiverSP slope2, plotted")
    if not valid_nodes.empty:
        ax.scatter(
            valid_nodes["node_s_m"].to_numpy(float) / 1000.0,
            valid_nodes["wse_ellipsoid"].to_numpy(float),
            s=22, marker="D", color=NEGATIVE, edgecolor="white", linewidth=0.45,
            zorder=6, label="RiverSP nodes",
        )

    ax.text(
        0.03, 0.96,
        f"reported slope: {riversp_reported_slope_m_per_km:.3f}\nreported slope2: {riversp_reported_slope2_m_per_km:.3f}\nfiltered: {row[filtered_col]:.3f}\nplotted RiverSP signs flipped",
        transform=ax.transAxes, ha="left", va="top", fontsize=7.2,
        bbox={"boxstyle": "round,pad=0.25", "facecolor": "white", "edgecolor": "#D5DCDD", "alpha": 0.92},
    )
    ax.set_title(f"Reach {row['reach_id']} | {row['obs_date_utc']}", fontsize=8.5, fontweight="bold", color=INK)
    ax.set_xlabel("Distance from downstream end (km)")
    ax.grid(color=GRID, linewidth=0.45)
    ax.spines[["top", "right"]].set_visible(False)

for ax in axes[::3]:
    ax.set_ylabel("Ellipsoidal WSE / height (m)")

# Keep one shared legend for the exploratory gallery.
handles, labels = axes[0].get_legend_handles_labels()
fig_profiles.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 0.965), ncol=7, frameon=False, fontsize=6.9)
fig_profiles.suptitle(
    "Exploratory profiles with opposite reported RiverSP and filtered signs",
    fontsize=11.5, fontweight="bold", color=INK,
)
plt.show()

example_table = pd.DataFrame([
    {
        "reach_id": row["reach_id"],
        "date": row["obs_date_utc"],
        "RiverSP slope": row["riversp_slope_m_per_km"],
        "filtered slope": row[filtered_col],
        "absolute sum": row["opposite_abs_resid"],
        "relative residual": row["opposite_rel_resid"],
        "file": row["file"],
    }
    for row, _, _ in examples
])
display(example_table)

print(
    "Interpretation note: RiverSP node WSEs are retrieved from Hydrocron and plotted as ellipsoidal height (wse + geoid_hght). "
    "The profile x-axis increases upstream from the downstream end, so reported RiverSP slopes are sign-flipped only for plotting lines on this axis. "
    "Opposite reported signs can therefore reflect coordinate/sign convention rather than an erroneous physical profile."
)


**Draft caption.** Figure 4. Slope-sign behavior of filtered PIXC in negative RiverSP cases. (a) Fraction of reach-overpasses with negative RiverSP *slope* or *slope2* for which filtered PIXC returned a positive downstream slope; percentages are conditional on the corresponding RiverSP estimate being negative. (b) Filtered PIXC slope as a function of RiverSP *slope* for reach-overpasses where RiverSP *slope* < 0. Teal points indicate positive filtered slopes and red points indicate filtered slopes that remain negative. (c) Distribution of filtered PIXC slopes for reach-overpasses where RiverSP *slope* is negative. The distribution includes both low-gradient positive slopes near zero and steeper positive slopes, reflecting the range of river gradients in the study domain. The display is limited to -5 to 20 m km$^{-1}$ and notes the five observations outside that range. Positive filtered slopes indicate improved physical plausibility and potential discharge-workflow usability, but do not alone establish absolute slope accuracy.